In [1]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

True

In [3]:
data = load_faq_data()
index = build_index(data)

In [4]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
)

response.output_text

"To determine if you can join a specific course, you'll need to check a few details, such as:\n\n1. **Enrollment Dates**: Look for information about when enrollment opens and closes.\n2. **Prerequisites**: Some courses may have prerequisites that you need to meet.\n3. **Availability**: Check if there are still spots available in the course.\n4. **Formats**: Determine if the course offers an online option or if it requires in-person attendance.\n5. **Institution Policies**: If it's offered through a university or specific institution, check their enrollment policies.\n\nIf you provide more context about the course you're referring to, I might be able to help with more specific information!"

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

### Create  a function tool schema

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course?"}', call_id='call_ZOcIhczNmsC2AwdyCSIVRrAK', name='search', type='function_call', id='fc_tmp_posa3wt77y', namespace=None, status='completed')]

In [9]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

### Now we send the result back to the LLM

In [10]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [11]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

"Yes, you can still join the course! However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. You don't need a confirmation email to start learning; you're already accepted and can begin working on the course materials and submitting homework."

### Calculating the ussage cost of the LLM

In [12]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(829, 58)

In [13]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(757, 53)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00014535


### Building the Agent Loop

In [14]:
# Helper function to handle function calls from the model
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

#### Make a single Call

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [16]:
#Send the call to the model with the instructions and the user question
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

#Check the response for function calls and handle them
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"Can I join the course"}


#### Write the full agent loop

In [17]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = client.responses.create(
        model="gpt-4o-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes, you can still join the course! However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted.

You don't need a confirmation email to start; you're accepted into the course and can begin learning and submitting homework anytime. Just remember that registration is mainly to gauge interest.

Feel free to start with the [LLM Zoomcamp documentation](https://datatalks.club/docs/courses/llm-zoomcamp/) for the learning materials.

Is there any other area or specific topic related to the course you would like to explore?


#### Wrap the whole agentic loop

In [18]:
def agent_loop(instructions, question, model="gpt-4o-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [19]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"run Olama locally"}
iteration #2...
ASSISTANT:
To run Olama locally, you can follow these steps:

1. **Install Olama**: 
   - Visit [Ollama's download page](https://ollama.com/download) and choose the appropriate installer for your operating system:
     - **macOS**: Download the `.pkg` file and install it.
     - **Windows**: Download the `.msi` file and install it.
     - **Linux**: Open your terminal and run the following command:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Run the LLaMA Model**:
   - After installation, open a terminal and execute:
     ```bash
     ollama run llama3
     ```
   - This command will download the LLaMA 3 model (approximately 4GB) and start it locally, opening a chat-like interface for interaction.

3. **Test the Local Server**: 
   - To ensure everything is working, run the following command:
     ```bash
     curl http://localhost:11434
     ```
   - You should 

'To run Olama locally, you can follow these steps:\n\n1. **Install Olama**: \n   - Visit [Ollama\'s download page](https://ollama.com/download) and choose the appropriate installer for your operating system:\n     - **macOS**: Download the `.pkg` file and install it.\n     - **Windows**: Download the `.msi` file and install it.\n     - **Linux**: Open your terminal and run the following command:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Run the LLaMA Model**:\n   - After installation, open a terminal and execute:\n     ```bash\n     ollama run llama3\n     ```\n   - This command will download the LLaMA 3 model (approximately 4GB) and start it locally, opening a chat-like interface for interaction.\n\n3. **Test the Local Server**: \n   - To ensure everything is working, run the following command:\n     ```bash\n     curl http://localhost:11434\n     ```\n   - You should receive a JSON response indicating the available models.\n\n4. **Insta

In [20]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"Can I join the course?"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! If you want to receive a certificate, please remember that you'll need to submit your project while the course is still accepting submissions. 

You don't need to register formally to start learning; you can begin anytime and submit your homework as long as the submission form is open. 

If you're looking for specific details on the course materials or workflow, feel free to ask! Are there any other areas you'd like to explore?


"Yes, you can still join the course! If you want to receive a certificate, please remember that you'll need to submit your project while the course is still accepting submissions. \n\nYou don't need to register formally to start learning; you can begin anytime and submit your homework as long as the submission form is open. \n\nIf you're looking for specific details on the course materials or workflow, feel free to ask! Are there any other areas you'd like to explore?"

In [21]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
function_call: search {"query":"Queen Gambit chess"}
iteration #3...
function_call: search {"query":"Queen Gambit definition"}
iteration #4...
function_call: search {"query": "Queen Gambit chess opening"}
function_call: search {"query": "Queen Gambit Netflix"}
function_call: search {"query": "Queen Gambit summary"}
iteration #5...
ASSISTANT:
The "Queen's Gambit" refers primarily to a popular chess opening and has also gained recognition from the Netflix miniseries that shares the same name.

1. **Queen's Gambit in Chess**: 
   - The Queen's Gambit is a chess opening that arises after the moves 1. d4 d5 2. c4. It is classified as a "gambit" because White offers a pawn on c4 to take control of the center of the board. This opening can lead to rich and complex positions. It often aims to create an advantage in space and development, while putting pressure on Black.

2. **Queen's Gambit Netflix Series**: 
   - T

'The "Queen\'s Gambit" refers primarily to a popular chess opening and has also gained recognition from the Netflix miniseries that shares the same name.\n\n1. **Queen\'s Gambit in Chess**: \n   - The Queen\'s Gambit is a chess opening that arises after the moves 1. d4 d5 2. c4. It is classified as a "gambit" because White offers a pawn on c4 to take control of the center of the board. This opening can lead to rich and complex positions. It often aims to create an advantage in space and development, while putting pressure on Black.\n\n2. **Queen\'s Gambit Netflix Series**: \n   - The phrase also refers to the critically acclaimed Netflix miniseries "The Queen\'s Gambit," which was released in 2020. The series follows the life of an orphaned chess prodigy, Beth Harmon, as she rises to prominence in the male-dominated world of chess during the Cold War era. It explores themes of genius, addiction, and the struggles of a young woman in pursuit of mastery.\n\nWould you like to explore more

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
It seems that "queen gambit" is not related to the course or its logistics, and therefore doesn't appear in the FAQ database. If you have any questions specifically related to the course or its logistics, please let me know! I'm here to help. 


'It seems that "queen gambit" is not related to the course or its logistics, and therefore doesn\'t appear in the FAQ database. If you have any questions specifically related to the course or its logistics, please let me know! I\'m here to help. '

In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [43]:
class OpenRouterClient:
    def __init__(self, base_url, api_key):
        self.client = OpenAI(
            base_url=base_url,
            api_key=api_key,
        )
    
    def send_request(self, chat_messages, tools=None, output_format=None):
        # Implement the expected interface
        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",  # Specify your model
            messages=chat_messages,
            tools=tools,
            # Handle output_format
        )
        return response

# Use it
client = OpenRouterClient(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

runner = OpenAIResponsesRunner(llm_client=client)

In [44]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [45]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [46]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

#### Creat the chat interface

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# Add the missing method
def send_request(self, chat_messages, tools=None, output_format=None):
    return self.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=chat_messages,
        tools=tools,
    )

OpenAI.send_request = send_request

# Now it will work
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
runner = OpenAIResponsesRunner(llm_client=client)

In [47]:
runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'